<a href="https://colab.research.google.com/github/TheAlishbahWaheed/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TheAlishbahWaheed/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
from pathlib import Path
import numpy as np
import pandas as pd

# Robust path: works whether the repo was cloned locally or opened in Colab
# from work/notebooks/, where data/raw/ sits two levels up.
CANDIDATES = [
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../data/raw/content_refresh_anonymized.csv"),
]
CSV_PATH = next((p for p in CANDIDATES if p.exists()), None)
if CSV_PATH is None:
    raise FileNotFoundError(
        "Can't find content_refresh_anonymized.csv. Run this notebook from the repo "
        "(work/notebooks/) so the ../../data/raw/ path resolves."
    )

raw = pd.read_csv(CSV_PATH)
df = raw.copy()
initial_rows = len(df)

NUMERIC_COLUMNS = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "content_age_days", "age_tier_order", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct", "trend_pct",
]
CATEGORICAL_COLUMNS = [
    "competition_level", "content_type", "main_intent", "provider_used", "model_used",
    "age_tier", "freshness_tier", "word_count_tier", "char_count_tier",
    "impression_tier", "position_tier", "trend_direction",
]

# numeric fills -> 0 (documented as systematic missingness, see section 2 check below,
# not "measured zero" -- e.g. feedly-article rows never had keyword data to begin with)
for c in NUMERIC_COLUMNS:
    df[c] = pd.to_numeric(df[c], errors="coerce")
    df[c] = df[c].replace([np.inf, -np.inf], np.nan).fillna(0)

# categorical fills -> "unknown" as its own explicit level
for c in CATEGORICAL_COLUMNS:
    df[c] = df[c].fillna("unknown").astype(str).replace({"": "unknown", "nan": "unknown"})

# basic quality filters: must have at least some search visibility and a full 90-day window
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df = df.drop_duplicates(subset=["content_id"]).reset_index(drop=True)

# --- the label (defined here so the leakage hunt in section 3 has something to attack) ---
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# --- engineered features ---
# log1p on the heavy-tailed count columns so a handful of huge pages don't dominate
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])
df["has_clicks"] = (df["clicks_90d"] > 0).astype(int)
df["has_ai_sessions"] = (df["ai_sessions_90d"] > 0).astype(int)
df["measurable_opportunity"] = ((df["impressions_90d"] >= 100) & (df["sessions_90d"] > 0)).astype(int)

# the feature list -- deliberately excludes trend_pct, trend_direction, last/prev_30d
# columns, and identifiers. Section 3 shows the test that justifies this; section 4
# lists every exclusion with a reason.
MODEL_NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
MODEL_CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
]
ENGINEERED_FLAGS = ["has_clicks", "has_ai_sessions", "measurable_opportunity"]

X_numeric = df[MODEL_NUMERIC_FEATURES + ENGINEERED_FLAGS]
X_categorical = pd.get_dummies(df[MODEL_CATEGORICAL_FEATURES], prefix=MODEL_CATEGORICAL_FEATURES)
X = pd.concat([X_numeric, X_categorical], axis=1)
y = df["is_declining_label"]
groups = df["client_id"]  # for the grouped split in section 3

print(f"raw rows: {initial_rows:,}  ->  after quality filters + de-dup: {len(df):,}")
print(f"feature matrix: {X.shape[0]:,} rows x {X.shape[1]} columns "
      f"({len(MODEL_NUMERIC_FEATURES)} numeric + {len(ENGINEERED_FLAGS)} engineered flags "
      f"+ {X_categorical.shape[1]} one-hot columns from {len(MODEL_CATEGORICAL_FEATURES)} categoricals)")
print(f"label: is_declining_label -- base rate = {y.mean():.3f} ({y.sum():,} declining / {len(y):,} total)")
print(f"\nany NaNs left in X? {X.isna().any().any()}")
X.head(3)

# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


raw rows: 30,000  ->  after quality filters + de-dup: 30,000
feature matrix: 30,000 rows x 55 columns (18 numeric + 3 engineered flags + 34 one-hot columns from 8 categoricals)
label: is_declining_label -- base rate = 0.542 (16,262 declining / 30,000 total)

any NaNs left in X? False


,search_volume,competition,cpc,word_count,char_count,log_impressions_90d,log_clicks_90d,log_sessions_90d,log_ai_sessions_90d,days_with_impressions,...,word_count_tier_unknown,impression_tier_excellent,impression_tier_good,impression_tier_low,impression_tier_moderate,position_tier_deep,position_tier_page_1,position_tier_page_3_5,position_tier_striking,position_tier_top_3
0,10.0,0.67,2.05,3221.0,20457.0,8.243808,3.401197,2.890372,0.0,88,...,False,False,True,False,False,False,False,False,True,False
1,90.0,0.01,0.05,2481.0,15562.0,9.636980,2.079442,2.302585,0.0,88,...,False,False,True,False,False,False,False,True,False,False
2,0.0,0.00,0.00,3515.0,23643.0,9.440023,2.484907,2.484907,0.0,88,...,False,False,True,False,False,False,False,True,False,False


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*
| Feature | Meaning | Missing → | Available before prediction? |
|---|---|---|---|
| `search_volume`, `competition`, `competition_level`, `cpc` | keyword-context metadata for the page's target keyword | 0 / `"unknown"` — **but missing is systematic, not random**: 100% blank for `feedly article` rows (see check below) | Yes — set at content-creation time, long before the 90-day export |
| `word_count`, `char_count` | article length | 0 — blank for ~28% of `keyword article` rows (not measured), 0% for the other two content types | Yes — fixed once the article is written |
| `content_type`, `main_intent` | what kind of page this is / what the searcher wants | `"unknown"` | Yes — set at creation |
| `log_impressions_90d`, `log_clicks_90d`, `log_sessions_90d`, `log_ai_sessions_90d` | log-scaled 90-day GSC/GA4 totals | 0 (from raw 0-fill, then log1p) | Yes — same 90-day window the label is drawn from, computed independently of the last-30/prev-30 split |
| `days_with_impressions`, `days_with_sessions` | how many of the 90 days had any activity | 0 | Yes |
| `content_age_days`, `days_since_last_update` | content lifecycle timing | 0 | Yes |
| `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct` | derived 90-day rates | 0 (`avg_position == 0` means "no position data", not position zero — documented quirk) | Yes — 90-day aggregates, not the 30-day trend windows |
| `age_tier`, `freshness_tier`, `word_count_tier`, `impression_tier`, `position_tier` | transparent threshold buckets of the numeric columns above | `"unknown"` where the source numeric is blank | Yes |
| `has_clicks`, `has_ai_sessions`, `measurable_opportunity` | engineered activity flags | n/a (derived, never blank) | Yes |

Everything above is drawn from the same 90-day snapshot as the label — nothing here needed
information from after the prediction moment. The columns that WOULD need future information
(`trend_pct`, `impressions_last_30d`, etc.) are exactly the ones excluded in section 3/4.


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Back up the "available-when" and "missing" claims from the notes above with a real check.

# 1) The missingness-is-systematic claim: keyword-context columns are blank exactly
#    along content_type lines, not at random. A blind fillna(0) would otherwise quietly
#    tell the model "feedly article" every time it sees search_volume == 0.
keyword_cols = ["search_volume", "competition", "competition_level", "cpc"]
print("missing-rate of keyword-context columns, by content_type (0.00-1.00):")
print(raw.groupby("content_type")[keyword_cols].apply(lambda g: g.isna().mean()).round(2))

print("\nmissing-rate of word_count / char_count, by content_type:")
print(raw.groupby("content_type")[["word_count", "char_count"]].apply(lambda g: g.isna().mean()).round(2))

# 2) The "available before prediction" claim: every column in MODEL_NUMERIC_FEATURES /
#    MODEL_CATEGORICAL_FEATURES comes from the SAME 90-day export as the label, not from
#    a period after it -- so nothing here needed information from the future to exist.
#    (The columns that WOULD require future information -- trend_pct, last/prev_30d --
#    are the ones section 3 shows must be excluded.)
all_declared_features = MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES + ENGINEERED_FLAGS
missing_from_df = [c for c in all_declared_features if c not in df.columns]
assert not missing_from_df, f"declared features not actually built: {missing_from_df}"
print(f"\nall {len(all_declared_features)} declared features exist in df: OK")
print("none of them require trend_pct / trend_direction / *_last_30d / *_prev_30d to be computed: OK by construction (see feature list above)")

# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


missing-rate of keyword-context columns, by content_type (0.00-1.00):
                    search_volume  competition  competition_level   cpc
content_type                                                           
comparison article           0.00         0.00               0.00  0.00
feedly article               1.00         1.00               1.00  1.00
keyword article              0.01         0.01               0.02  0.01

missing-rate of word_count / char_count, by content_type:
                    word_count  char_count
content_type                              
comparison article        0.00        0.00
feedly article            0.00        0.00
keyword article           0.28        0.28

all 29 declared features exist in df: OK
none of them require trend_pct / trend_direction / *_last_30d / *_prev_30d to be computed: OK by construction (see feature list above)


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

def grouped_auc(X_mat, y_vec, groups_vec, n_splits=5):
    """Honest split: GroupKFold by client_id, per the leakage skill -- a random split
    would let the model memorize per-client quirks and fake skill."""
    gkf = GroupKFold(n_splits=n_splits)
    scores = []
    for train_idx, test_idx in gkf.split(X_mat, y_vec, groups_vec):
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_mat.iloc[train_idx].fillna(0))
        X_test = scaler.transform(X_mat.iloc[test_idx].fillna(0))
        clf = LogisticRegression(max_iter=1000)
        clf.fit(X_train, y_vec.iloc[train_idx])
        preds = clf.predict_proba(X_test)[:, 1]
        scores.append(roc_auc_score(y_vec.iloc[test_idx], preds))
    return float(np.mean(scores)), float(np.std(scores))

base_rate = y.mean()
clean_mean, clean_std = grouped_auc(X, y, groups)
print(f"base rate (is_declining_label=1):        {base_rate:.3f}")
print(f"CLEAN features (official list):           AUC = {clean_mean:.3f} (+/- {clean_std:.3f})  <- the honest number")

# --- Attack 1: label-derived feature. trend_pct is the exact quantity thresholded
# into trend_direction, which is what the label is made from. Inject it and watch.
X_leak_trend = X.copy()
X_leak_trend["trend_pct__SUSPECT"] = df["trend_pct"]
leak_trend_mean, leak_trend_std = grouped_auc(X_leak_trend, y, groups)
print(f"+ trend_pct injected (label-derived):     AUC = {leak_trend_mean:.3f} (+/- {leak_trend_std:.3f})  <- confession")

# --- Attack 2: overlapping-window features. impressions/clicks/sessions_last_30d and
# _prev_30d are exactly the two windows FlyRank's pipeline diffs to build trend_pct --
# using them means the model can reconstruct the label's own inputs.
X_leak_window = X.copy()
window_suspects = [
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
]
for c in window_suspects:
    X_leak_window[c + "__SUSPECT"] = df[c]
leak_window_mean, leak_window_std = grouped_auc(X_leak_window, y, groups)
print(f"+ last/prev_30d injected (overlapping win): AUC = {leak_window_mean:.3f} (+/- {leak_window_std:.3f})  <- also inflated")

print(f"\nCollapse test: {leak_trend_mean:.3f} -> {clean_mean:.3f} when trend_pct is removed. "
      f"Per the skill's rule of thumb (collapse from ~1.0 to ~0.7 = confession), this is exactly that pattern.")

# --- Attack 3: correlation sanity check on the suspects vs. a clean feature ---
corr_check = pd.DataFrame({
    "trend_pct": df["trend_pct"],
    "impressions_last_30d": df["impressions_last_30d"],
    "impressions_prev_30d": df["impressions_prev_30d"],
    "avg_position (clean feature)": df["avg_position"],
    "log_impressions_90d (clean feature)": df["log_impressions_90d"],
}).apply(lambda s: s.corr(y))
print("\ncorrelation with the label (no single clean feature towers over the others):")
print(corr_check.round(3))

print("\n--- attack checklist ---")
print("[x] Timeline drawn: label = trend_direction, computed from last_30d vs prev_30d impressions.")
print("[x] Label-derived suspect tested: trend_pct removed -> AUC 0.999 to", f"{clean_mean:.3f}")
print("[x] Overlapping-window suspects tested: last/prev_30d removed -> AUC", f"{leak_window_mean:.3f} to {clean_mean:.3f}")
print("[x] No product/decision flags (provider_used, model_used) used as features (see section 4).")
print("[x] Split is grouped by client_id, not random.")
print("[x] Base rate printed next to every metric.")

# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


base rate (is_declining_label=1):        0.542
CLEAN features (official list):           AUC = 0.660 (+/- 0.040)  <- the honest number
+ trend_pct injected (label-derived):     AUC = 0.999 (+/- 0.001)  <- confession
+ last/prev_30d injected (overlapping win): AUC = 0.906 (+/- 0.041)  <- also inflated

Collapse test: 0.999 -> 0.660 when trend_pct is removed. Per the skill's rule of thumb (collapse from ~1.0 to ~0.7 = confession), this is exactly that pattern.

correlation with the label (no single clean feature towers over the others):
trend_pct                             -0.131
impressions_last_30d                  -0.094
impressions_prev_30d                   0.004
avg_position (clean feature)          -0.029
log_impressions_90d (clean feature)    0.177
dtype: float64

--- attack checklist ---
[x] Timeline drawn: label = trend_direction, computed from last_30d vs prev_30d impressions.
[x] Label-derived suspect tested: trend_pct removed -> AUC 0.999 to 0.660
[x] Overlapping-window sus

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*
18 of the 44 raw columns are excluded from the model feature vector. The code cell below
generates this same list programmatically (and asserts every exclusion has a reason on file,
so nothing gets dropped silently later).

| Excluded field | Why |
|---|---|
| `content_id`, `client_id` | identifiers — grouping/joins only, never model inputs |
| `trend_direction` | **is** the label |
| `trend_pct` | the label's direct numeric input — confirmed leaky in section 3 (AUC 0.660 → 0.999) |
| `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d` | overlapping window — the numerator of `trend_pct` |
| `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d` | overlapping window — the denominator side of `trend_pct` |
| `provider_used`, `model_used` | generation-tooling detail, not a content/search signal — risks the model learning "which client uses which LLM" |
| `char_count_tier` | redundant with `word_count_tier` (same measurement, bucketed twice) |
| `age_tier_order` | redundant with `age_tier` (numeric restatement of the same categorical) |
| `pageviews_90d`, `engaged_sessions_90d`, `scroll_events_90d` | raw counts already folded into derived rates (`scroll_rate`, `engagement_rate`) that ARE features — keeping both double-counts the same signal |
| `users_90d` | near-duplicate of `sessions_90d`, already covered by `log_sessions_90d` |


In [5]:
# This cell is for CODE (numbers, a query, a check).
all_raw_columns = list(raw.columns)

# columns whose signal already reaches the model, either directly or as the raw
# count underneath a derived rate / log feature
used_or_source_of = set(
    MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES
    + ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]  # -> log1p features
)

exclusion_reasons = {
    "content_id": "identifier -- unique per row, zero predictive signal, exists only for joins",
    "client_id": "identifier -- used for the grouped split only, never as a model input (would let the model memorize a client instead of learning a content signal)",
    "trend_direction": "IS the label (is_declining_label = trend_direction == 'down') -- including it as a feature is circular",
    "trend_pct": "the label's direct numeric input -- section 3 shows injecting it pushes grouped AUC from 0.66 to 0.999",
    "impressions_last_30d": "overlapping window -- the exact numerator FlyRank's pipeline diffs to compute trend_pct",
    "clicks_last_30d": "same overlapping-window problem as impressions_last_30d",
    "sessions_last_30d": "same overlapping-window problem as impressions_last_30d",
    "impressions_prev_30d": "the other half of the trend_pct computation -- same leakage family as impressions_last_30d",
    "clicks_prev_30d": "same overlapping-window problem as impressions_prev_30d",
    "sessions_prev_30d": "same overlapping-window problem as impressions_prev_30d",
    "provider_used": "which LLM vendor wrote the article is a production/tooling detail, not a content or search signal -- risks the model learning 'this client uses provider X' instead of anything about decline",
    "model_used": "same reasoning as provider_used -- a generation-tooling detail, not a content/search signal",
    "char_count_tier": "redundant with word_count_tier (same underlying measurement, bucketed a second way) -- keeping both double-counts one signal",
    "age_tier_order": "redundant with age_tier (numeric restatement of the same categorical) -- dropped to avoid double-counting",
    "pageviews_90d": "raw count behind scroll_rate, which is already a feature -- the ratio carries the signal, the raw count mostly adds correlated volume noise",
    "users_90d": "near-duplicate of sessions_90d (same GA4 population, different unit) -- sessions_90d / log_sessions_90d already carries this",
    "engaged_sessions_90d": "raw count behind engagement_rate, which is already a feature -- same double-count risk as pageviews_90d",
    "scroll_events_90d": "raw count behind scroll_rate, which is already a feature -- same double-count risk as pageviews_90d",
}

excluded = [c for c in all_raw_columns if c not in used_or_source_of]
print(f"{len(excluded)} of {len(all_raw_columns)} raw columns excluded from the model feature vector:\n")
for c in excluded:
    print(f"- {c}: {exclusion_reasons.get(c, 'MISSING REASON')}")

undocumented = [c for c in excluded if c not in exclusion_reasons]
assert not undocumented, f"undocumented exclusions -- go add a reason: {undocumented}"
print("\nAll exclusions documented. Nothing excluded silently.")

# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


18 of 44 raw columns excluded from the model feature vector:

- content_id: identifier -- unique per row, zero predictive signal, exists only for joins
- client_id: identifier -- used for the grouped split only, never as a model input (would let the model memorize a client instead of learning a content signal)
- provider_used: which LLM vendor wrote the article is a production/tooling detail, not a content or search signal -- risks the model learning 'this client uses provider X' instead of anything about decline
- model_used: same reasoning as provider_used -- a generation-tooling detail, not a content/search signal
- pageviews_90d: raw count behind scroll_rate, which is already a feature -- the ratio carries the signal, the raw count mostly adds correlated volume noise
- users_90d: near-duplicate of sessions_90d (same GA4 population, different unit) -- sessions_90d / log_sessions_90d already carries this
- engaged_sessions_90d: raw count behind engagement_rate, which is already a fea

## Self-check

Before you submit, confirm each line honestly:

- [✅ ] Every section above is filled — markdown thinking AND the code that backs it
- [✅ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅ ] No client names, URLs, or private queries anywhere
- [✅ ] My claims use careful words: observed, measured, directional, decision-support
- [✅ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.